# pima: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('01_pima_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,69.170,61.680,72.253,54.060,88.8,71.430,71.100,82.593,65.425
smote,73.058,65.391,63.615,67.655,79.3,73.478,69.477,82.497,69.224
bsmote,73.569,66.069,60.373,73.445,74.1,73.773,68.284,82.094,69.819
ros,73.242,65.633,63.095,68.651,78.3,73.476,71.178,83.210,69.437
adasyn,72.996,65.447,64.052,68.218,79.0,73.609,69.591,82.287,69.222
casb,72.877,65.341,64.005,68.043,78.9,73.472,71.531,83.266,69.109
casb_relaxed,71.985,64.269,65.193,64.658,81.2,72.929,69.798,82.632,68.127
casb_edited,75.254,68.263,60.521,79.224,72.1,75.662,69.793,82.687,71.759
casb_ros,72.310,64.799,65.881,64.710,81.5,73.105,72.082,83.501,68.555


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,68.481,60.143,65.352,55.933,84.4,70.166,65.808,79.615,64.312
smote,69.922,61.767,57.850,67.296,73.6,70.448,61.788,78.370,65.845
bsmote,70.205,62.379,56.333,71.202,70.3,70.751,61.630,78.520,66.292
ros,71.233,63.221,60.034,67.327,75.8,71.564,64.070,78.922,67.227
adasyn,69.238,61.052,57.053,67.124,72.6,69.862,61.629,78.044,65.145
casb,71.143,63.095,58.862,68.253,74.4,71.326,63.299,78.953,67.119
casb_relaxed,71.585,63.628,60.646,67.509,76.4,71.954,64.658,79.407,67.606
casb_edited,71.686,64.356,56.882,75.189,69.0,72.094,63.545,79.317,68.021
casb_ros,70.062,61.737,58.212,66.003,74.7,70.351,63.564,78.653,65.899


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,71.487,64.069,69.233,59.850,85.6,72.725,69.591,82.774,67.778
smote,73.392,65.879,64.764,67.484,80.2,73.842,67.567,82.074,69.636
bsmote,73.914,66.412,63.487,69.909,78.4,74.155,68.101,82.196,70.163
ros,73.615,66.126,65.509,66.922,81.2,74.061,69.310,82.602,69.870
adasyn,73.982,66.460,63.003,70.489,77.8,74.145,66.883,81.681,70.221
casb,73.098,65.496,64.599,66.565,80.4,73.483,69.690,83.060,69.297
casb_relaxed,74.526,67.209,65.114,69.574,79.9,74.737,70.166,83.237,70.868
casb_edited,75.654,68.763,61.025,79.423,72.5,75.962,70.462,83.267,72.209
casb_ros,73.389,65.939,65.826,66.184,81.5,73.842,69.547,82.891,69.664


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,70.768,63.416,71.135,57.442,87.4,72.421,71.694,83.175,67.092
smote,73.152,65.712,63.932,68.623,78.6,73.612,71.462,82.995,69.432
bsmote,73.285,65.912,63.986,69.001,78.5,73.750,72.330,83.257,69.598
ros,73.295,65.843,64.336,68.249,79.2,73.725,71.894,83.240,69.569
adasyn,73.684,66.279,64.760,68.449,79.7,74.074,72.200,83.137,69.982
casb,72.613,65.088,65.230,65.832,80.7,73.266,72.060,83.253,68.850
casb_relaxed,73.308,65.806,64.243,68.242,79.3,73.771,72.047,83.330,69.557
casb_edited,72.315,65.411,59.538,76.038,70.7,73.369,73.110,83.615,68.863
casb_ros,72.912,65.377,64.791,66.758,80.2,73.479,72.127,83.281,69.144


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,69.976,62.327,69.493,56.821,86.550,71.686,69.548,82.039,66.152
smote,72.381,64.687,62.540,67.765,77.925,72.845,67.573,81.484,68.534
bsmote,72.743,65.193,61.045,70.889,75.325,73.107,67.586,81.517,68.968
ros,72.846,65.206,63.244,67.787,78.625,73.206,69.113,81.993,69.026
adasyn,72.475,64.810,62.217,68.570,77.275,72.923,67.576,81.287,68.642
casb,72.433,64.755,63.174,67.173,78.600,72.887,69.145,82.133,68.594
casb_relaxed,72.851,65.228,63.799,67.496,79.200,73.348,69.167,82.152,69.040
casb_edited,73.727,66.699,59.491,77.469,71.075,74.272,69.228,82.221,70.213
casb_ros,72.168,64.463,63.677,65.914,79.475,72.694,69.330,82.082,68.316


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,75.254,bsmote,73.569,1.685,WIN
1,f1,casb_edited,68.263,bsmote,66.069,2.195,WIN
2,precision,casb_ros,65.881,adasyn,64.052,1.829,WIN
3,recall,casb_edited,79.224,bsmote,73.445,5.779,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,71.686,ros,71.233,0.453,WIN
1,f1,casb_edited,64.356,ros,63.221,1.136,WIN
2,precision,casb_relaxed,60.646,ros,60.034,0.612,WIN
3,recall,casb_edited,75.189,bsmote,71.202,3.987,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,75.654,adasyn,73.982,1.672,WIN
1,f1,casb_edited,68.763,adasyn,66.460,2.304,WIN
2,precision,casb_ros,65.826,ros,65.509,0.317,WIN
3,recall,casb_edited,79.423,adasyn,70.489,8.934,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,73.727,ros,72.846,0.881,WIN
1,f1,casb_edited,66.699,ros,65.206,1.493,WIN
2,precision,casb_relaxed,63.799,ros,63.244,0.555,WIN
3,recall,casb_edited,77.469,bsmote,70.889,6.579,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.734343,0.673684,0.780488,0.592593,0.91,0.751296,0.749657,0.842593,0.704013,None
1,42,1,svm,smote,0.75,0.754983,0.684685,0.666667,0.703704,0.81,0.756852,0.750266,0.840741,0.719834,None
2,42,1,svm,bsmote,1.00,0.751911,0.682540,0.597222,0.796296,0.71,0.753148,0.681104,0.818704,0.717225,None
3,42,1,svm,ros,0.75,0.748331,0.679245,0.692308,0.666667,0.84,0.753333,0.753929,0.846667,0.713788,None
4,42,1,svm,adasyn,0.75,0.755351,0.684211,0.650000,0.722222,0.79,0.756111,0.742967,0.836296,0.719781,None
5,42,1,svm,casb,0.75,0.754124,0.685185,0.685185,0.685185,0.83,0.757593,0.765311,0.849444,0.719655,None
6,42,1,svm,casb_relaxed,1.00,0.750309,0.677966,0.625000,0.740741,0.76,0.750370,0.712230,0.841852,0.714137,None
7,42,1,svm,casb_edited,0.75,0.779957,0.714286,0.625000,0.833333,0.73,0.781667,0.696521,0.831111,0.747121,None
8,42,1,svm,casb_ros,0.75,0.749568,0.678899,0.672727,0.685185,0.82,0.752593,0.763594,0.850185,0.714233,None
9,42,1,knn,original,0.50,0.716473,0.640777,0.673469,0.611111,0.84,0.725556,0.694364,0.824352,0.678625,None
